# Week 37: Structure Span Prediction

In [ ]:
import copy
import torch
import nb_setup
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from src.qa_dataset import QADataset
from src.utils import build_vocabulary
from torch.utils.data import DataLoader
from src.data import load_tokenised_data
from src.training import train_one_epoch
from src.evaluation import evaluate_model
from torch.nn.utils.rnn import pad_sequence
from src.bio_conversion import create_bio_labels, test_bio_conversion
from src.labeller import SequenceLabeller, EmptySpanBaseline, LocalSequenceLabeller

In [ ]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

### BIO Labelling and Validation

In [ ]:
bio_labels_train = df_train_tokenised.apply(
    lambda row: create_bio_labels(
        row["answerable"],
        row["context"],
        row["context_tokens"],
        row["context_offsets"],
        row["answer_start"],
        row["answer"],
    ),
    axis=1,
)
df_train_tokenised["bio_labels"], df_train_tokenised["bio_alignment"] = zip(*bio_labels_train)

bio_labels_val = df_val_tokenised.apply(
    lambda row: create_bio_labels(
        row["answerable"],
        row["context"],
        row["context_tokens"],
        row["context_offsets"],
        row["answer_start"],
        row["answer"],
    ),
    axis=1,
)
df_val_tokenised["bio_labels"], df_val_tokenised["bio_alignment"] = zip(*bio_labels_val)

In [ ]:
df_train_tokenised["bio_alignment"].value_counts()

In [ ]:
test_bio_conversion()

### Question-conditioned Sequence Labeller

In [ ]:
vocabulary = build_vocabulary(df_train_tokenised)

In [ ]:

def collate_fn(batch):
    question_ids = pad_sequence(
        [item["question_ids"] for item in batch],
        batch_first=True,
        padding_value=0,
    )

    context_ids = pad_sequence(
        [item["context_ids"] for item in batch],
        batch_first=True,
        padding_value=0,
    )

    labels = pad_sequence(
        [item["labels"] for item in batch],
        batch_first=True,
        padding_value=-100,
    )

    return {
        "question_ids": question_ids,
        "context_ids": context_ids,
        "labels": labels,
        "lang": [item["lang"] for item in batch],
    }



train_dataset = QADataset(
    df_train_tokenised,
    vocabulary,
)

val_dataset = QADataset(
    df_val_tokenised,
    vocabulary,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

### Neural Network

In [ ]:
NUM_EPOCHS = 40

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
model = SequenceLabeller(
    vocab_size=len(vocabulary),
    embedding_dim=300,
    hidden_dim=64,
    num_labels=3,
    padding_idx=0,
).to(device)

# model = GRUSequenceLabeller(
#     vocab_size=len(vocabulary),
# ).to(device)

class_weights = torch.tensor(
    [1.0, 5.0, 3.0],
    dtype=torch.float,
    device=device,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
)

best_val_f1 = -1
best_model_state = None

for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )

    sequence_results = evaluate_model(
        model,
        val_loader,
        device,
        model_name="Sequence Labeller",
    )
    val_f1 = sequence_results[sequence_results["Language"] == "Overall"]["Token F1"].values[0]
    val_precision = sequence_results[sequence_results["Language"] == "Overall"]["Precision"].values[0]
    val_recall = sequence_results[sequence_results["Language"] == "Overall"]["Recall"].values[0]

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f}"
    )

In [ ]:
model.load_state_dict(best_model_state)

sequence_results = evaluate_model(
    model,
    val_loader,
    device,
    model_name="Sequence Labeller",
)

baseline = EmptySpanBaseline().to(device)

baseline_results = evaluate_model(
    baseline,
    val_loader,
    device,
    model_name="Empty Baseline",
)

results_df = pd.concat(
    [sequence_results, baseline_results],
    ignore_index=True,
)

### Logistic Regression

In [ ]:
lr_model = LocalSequenceLabeller(
    class_weight={
        0: 1.0,
        1: 3.0,
        2: 2.0,
    }
)

lr_model.fit(train_loader)

In [ ]:
lr_results = evaluate_model(
    lr_model,
    val_loader,
    device,
    model_name="Local Logistic Regression",
)

results_df = pd.concat(
    [results_df, lr_results],
    ignore_index=True,
)

In [ ]:
results_df[
    ["Precision", "Recall", "Token F1", "Exact Span Match"]
] = results_df[
    ["Precision", "Recall", "Token F1", "Exact Span Match"]
].round(4)
results_df